# Monte Carlo credit-loss simulation

Adnan Masood · University of South Florida

Estimate possible losses across a fictional portfolio of loans. The computer draws many possible default patterns using stated assumptions, calculates each portfolio loss, and shows the average, large losses, and how often losses cross a reserve amount.

**Dataset:** There are 1,000 equal loans and 2,500 simulated scenarios. For each loan, the assumed default chance (PD) is 2.5%, the amount owed (EAD) is $10,000, and the lost share after default (LGD) is 45%. Seed 17 makes the random draws repeatable. The dependence setting allows borrowers’ defaults to move together; it is a model assumption, not observed proof.

**Task:** Calculate average loss, the 95th and 99th percentiles (loss levels reached or exceeded in roughly 5% and 1% of scenarios), the fraction above a $200,000 reserve, and the standard error for that estimated fraction. Standard error describes how much the fraction may wobble because only 2,500 random scenarios were run; it does not measure whether the assumptions are right.

**Compare:** For the same loans, compare the simulated average with the arithmetic benchmark: 1,000 × 2.5% × $10,000 × 45% = $112,500. Then set the dependence input to zero while keeping the loan assumptions, path count, and seed fixed. Compare the average with the largest losses; defaults moving together can change the tail without changing the individual default chances.

**Runtime:** NumPy generates the random outcomes and SciPy provides the normal-distribution calculations. More paths generally make random sampling wiggle less, but cannot validate the 2.5% default chance, 45% loss share, or Gaussian dependence assumption. A precise answer under a wrong assumption is still misleading.


In [ ]:
import numpy as np
from scipy.stats import norm
# Seed 17 repeats the random draws; each row below will be one possible portfolio outcome.
rng = np.random.default_rng(17)
n_sims,n_loans = 2500,1000
# PD is chance of default, EAD is amount owed, and LGD is the share lost after default.
pd = np.full(n_loans,.025)
ead = np.full(n_loans,10000.0)
lgd = np.full(n_loans,.45)
# rho controls a shared shock: positive values make defaults tend to occur together.
rho = .12
common = rng.normal(size=(n_sims,1))
idiosyncratic = rng.normal(size=(n_sims,n_loans))
latent = np.sqrt(rho)*common+np.sqrt(1-rho)*idiosyncratic
defaults = latent < norm.ppf(pd)
loss = (defaults*ead*lgd).sum(axis=1)
# Compare the simulated average with sum(PD × amount owed × lost share).
print("Expected loss:",loss.mean())
print("Analytical expected-loss baseline:",(pd*ead*lgd).sum())
print("95th percentile:",np.quantile(loss,.95))
print("99th percentile:",np.quantile(loss,.99))
breach = (loss>200000).mean()
print("P(loss > $200k):",breach)
print("Simulation SE:",np.sqrt(breach*(1-breach)/n_sims))


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Set latent correlation to zero

Compare average loss and the tail while keeping marginal PDs fixed.

Replace:
```python
rho = .12
```
With:
```python
rho = 0
```

### Double default probabilities

Compare expected loss, tail quantiles, and reserve breaches with the initial PD assumption.

Replace:
```python
n_loans,.025
```
With:
```python
n_loans,.050
```

### Double simulation paths

Compare the breach estimate and its simulation standard error under unchanged risk assumptions.

Replace:
```python
n_sims,n_loans = 2500,1000
```
With:
```python
n_sims,n_loans = 5000,1000
```



In [ ]:
# Compare rho=0.12 with rho=0 using the same number of paths and seed.
# Record mean loss, both quantiles, breach probability, and its simulation SE.


## Break it

Double each loan's PD from 2.5% to 5%. Compare mean and tail losses with the initial run. Then double the paths to distinguish changed risk assumptions from reduced simulation noise.


In [ ]:
# Run PD=0.05, then compare 2,500 and 5,000 paths at that same PD.
# Explain why a smaller simulation SE does not confirm the assumed PD.


## What to submit

- The simulated average and $112,500 arithmetic benchmark, both loss percentiles, reserve-breach fraction, and its simulation standard error.
- Results after changing dependence and default probability. State which inputs changed and whether the same random seed and path count were used.
- A plain-English distinction between random sampling wobble and uncertainty about the default chances, lost amount, or whether borrowers fail together.

## Optional extension

Exposure and the lost share are fixed for each loan here, so the expected-loss multiplication is direct. If loan amounts or recoveries also varied, their relationship to default would need explicit assumptions; multiplying separate averages might then give the wrong answer.
